In [2]:
import langchain_tavily
import certifi
import asyncio
import os
from dotenv import load_dotenv
import ssl
import json
from datetime import datetime
from langchain_tavily import TavilyMap, TavilyExtract
from typing import List, Dict, Any

from rich.console import Console
from rich.table import Table
from rich.progress import Progress
from rich.panel import Panel


In [3]:
#Configure ssl context to use certifi's CA bundle for secure connections
ssl_context = ssl.create_default_context(cafile=certifi.where())
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
console=Console()
print("All imported successfully")

All imported successfully


In [4]:

load_dotenv()

True

### TavilyMap: Website Structure Discovery
Tavily map automatically discovers and maps website structures by crawling through links. Its perfect for:-
- Document sites
- Blog archives
- Knowledge bases
- Any structured websites

#### Key params
- max_depth :- How deep to crawl
- max_breadth :- How many links in page
- max_pages :- Max pages to cover

In [5]:
tavily_map=TavilyMap(
    max_depth=3,
    max_breadth=15,
    max_pages=500
) # gets links available in the site


In [6]:
url= "https://docs.python.org/3/library/asyncio.html"
site_map = tavily_map.invoke(url)

urls=site_map.get('results', [] )
console.print(f"Total URLs found: {len(urls)}")

console.print("Sample URLs:", style="bold green")
for url in urls[:10]:  # Print first 10 URLs as a sample
    console.print(f"- {url}", style="bold green")

Total URLs found: 56

Sample URLs:

- https://docs.python.org/3/library/asyncio.html

- https://docs.python.org/3/library/codecs.html

- https://docs.python.org/3/library/asyncio-protocol.html

...

- https://docs.python.org/3/library/signal.html

- https://docs.python.org/3/copyright.html

- https://docs.python.org/3/library/site.html

- https://docs.python.org/3/library/asyncio-platforms.html

- https://docs.python.org/license.html

In [6]:
tavily_extract=TavilyExtract()

In [ ]:
sample_urls=urls[:10]  # Extract content from the first 10 URLs as a sample

extraction_results = await tavily_extract.ainvoke(input={"urls": sample_urls})

extracted_docs=extraction_results.get('results', [])

for i,doc in enumerate(extracted_docs):
    url=doc.get('url', 'N/A')
    content=doc.get('raw_content', 'N/A')
    panel_content=f"""
        URL: {url}
        Preview: {content[:200]}...  # Show only the first 200 characters as a preview
            """
    console.print(Panel(panel_content, title=f"Document {i+1}", expand=False))

In [ ]:
def chunk_urls(urls: List[str], chunk_size: int) -> List[List[str]]:
    """Utility function to split URLs into chunks."""
    return [urls[i:i + chunk_size] for i in range(0, len(urls), chunk_size)]

async def extract_batch(url_batch:List[str], batch_num:int) -> List[Dict[str, Any]]:
    """Extract content for a batch of URLs."""
    try:
        result = await tavily_extract.ainvoke(input={"urls": url_batch})
        return result.get('results', []) 
    except Exception as e:
        console.print(f"Error extracting batch {batch_num}: {e}")
        return []
url_chunks = chunk_urls(urls, chunk_size=10)  # Adjust chunk size as needed

# Process batches concurrently using asyncio.gather
tasks = [extract_batch(chunk, idx) for idx, chunk in enumerate(url_chunks)]
batch_results = await asyncio.gather(*tasks, return_exceptions=True)

#Flatten the list of lists into a single list of documents
all_processed_docs = []
failed_batches = []
for result in batch_results:
    if isinstance(result, Exception):
        failed_batches.append(result)
    else:
        for doc in result["results"]:
            doc=Document(
                page_content=doc.get('raw_content', ''),
                metadata={"source": doc.get('url', '')}
            )
            all_processed_docs.append(doc)
